In [6]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
import io
import requests

# Load the data from URL using openpyxl for xlsx format
url = "https://www.nyc.gov/assets/finance/downloads/pdf/rolling_sales/annualized-sales/2024/2024_manhattan.xlsx"
response = requests.get(url)
df = pd.read_excel(io.BytesIO(response.content), skiprows=3, engine='openpyxl')
df.columns = df.columns.str.strip().str.upper()
df = pd.read_excel("2024_manhattan.xlsx", skiprows=3, engine='openpyxl')





In [7]:
# Clean & convert columns
price_cols = ['AVERAGE SALE PRICE', 'MEDIAN SALE PRICE', 'MAXIMUM SALE PRICE', 'NUMBER OF SALES']
for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['NEIGHBORHOOD', 'AVERAGE SALE PRICE', 'NUMBER OF SALES'])

# Create new liquidity metric
df['LIQUIDITY'] = df['AVERAGE SALE PRICE'] * df['NUMBER OF SALES']

# Dash app
app = Dash(__name__)
app.title = "NYC Real Estate Advisory 2024"

# Layout
app.layout = html.Div([
    html.H1("NYC Manhattan 2024 Real Estate Market Dashboard", style={'textAlign': 'center'}),

    dcc.Dropdown(
        id='price_type',
        options=[
            {'label': 'Average Sale Price', 'value': 'AVERAGE SALE PRICE'},
            {'label': 'Median Sale Price', 'value': 'MEDIAN SALE PRICE'},
            {'label': 'Maximum Sale Price', 'value': 'MAXIMUM SALE PRICE'}
        ],
        value='AVERAGE SALE PRICE',
        style={'width': '60%', 'margin': 'auto'}
    ),

    html.Div([
        dcc.Graph(id='price_by_neighborhood'),
        dcc.Graph(id='sales_volume'),
        dcc.Graph(id='liquidity_plot'),
        dcc.Graph(id='affordable_growth')
    ])
])

# Callback
@app.callback(
    [Output('price_by_neighborhood', 'figure'),
     Output('sales_volume', 'figure'),
     Output('liquidity_plot', 'figure'),
     Output('affordable_growth', 'figure')],
    [Input('price_type', 'value')]
)
def update_graphs(price_type):
    fig_price = px.bar(df, x='NEIGHBORHOOD', y=price_type,
                       title=f"{price_type} by Neighborhood",
                       labels={price_type: price_type}, height=400)

    fig_sales = px.bar(df, x='NEIGHBORHOOD', y='NUMBER OF SALES',
                       title="Number of Sales by Neighborhood", height=400)

    fig_liquidity = px.scatter(df, x='NUMBER OF SALES', y='AVERAGE SALE PRICE',
                               size='LIQUIDITY', color='NEIGHBORHOOD',
                               title="Liquidity: Sales Volume x Average Price")

    fig_affordable = px.scatter(df, x='MEDIAN SALE PRICE', y='NUMBER OF SALES',
                                color='NEIGHBORHOOD', size='NUMBER OF SALES',
                                title="Affordable Growth Potential: Sales vs. Median Price")

    return fig_price, fig_sales, fig_liquidity, fig_affordable

# Run app
if __name__ == '__main__':
    app.run(debug=True)